In [0]:
-- Top 3 customers by total spending in each category,
-- using ROW_NUMBER() partitioned by category

WITH customer_category_spending AS (
  SELECT
    c.category_name,
    o.customer_id,
    SUM(o.amount) AS total_spent
  FROM orders o
  JOIN order_items oi
    ON o.order_id = oi.order_id
  JOIN products p
    ON oi.product_id = p.product_id
  JOIN categories c
    ON p.category_id = c.category_id
  GROUP BY c.category_name, o.customer_id
),

ranked AS (
  SELECT
    category_name,
    customer_id,
    total_spent,
    ROW_NUMBER() OVER (
      PARTITION BY category_name
      ORDER BY total_spent DESC
    ) AS spend_rank
  FROM customer_category_spending
)

SELECT
  category_name,
  customer_id,
  total_spent,
  spend_rank
FROM ranked
WHERE spend_rank <= 3
ORDER BY category_name, spend_rank;

In [0]:
-- Employees who earn more than their manager
-- (self-join employees on manager_id)

SELECT
  e.employee_id,
  e.name AS employee_name,
  e.salary AS employee_salary,
  m.employee_id AS manager_id,
  m.name AS manager_name,
  m.salary AS manager_salary
FROM employees e
JOIN employees m
  ON e.manager_id = m.employee_id
WHERE e.salary > m.salary
ORDER BY e.salary DESC;

In [0]:
-- Consecutive login days: group each user's login dates into "islands" of
-- consecutive days using the date - ROW_NUMBER() trick, then get streak bounds.

WITH logins AS (
  -- one row per user per day (dedupe multiple logins on the same day)
  SELECT
    user_id,
    CAST(login_ts AS DATE) AS login_date
  FROM user_logins
  GROUP BY user_id, CAST(login_ts AS DATE)
),

streaks AS (
  SELECT
    user_id,
    login_date,
    -- For consecutive days, login_date - row_number() is constant
    -- within each streak (any unit works; here we subtract via date_id)
    DATE_SUB(login_date, ROW_NUMBER() OVER (
      PARTITION BY user_id
      ORDER BY login_date
    )) AS streak_key
  FROM logins
)

SELECT
  user_id,
  MIN(login_date) AS streak_start,
  MAX(login_date) AS streak_end,
  COUNT(*) AS total_days
FROM streaks
GROUP BY user_id, streak_key
HAVING COUNT(*) >= 2   -- keep only true streaks (2+ consecutive days); drop this to see all
ORDER BY user_id, streak_start;

In [0]:
-- Cumulative monthly sales per product:
-- 1) aggregate revenue to one row per product per month,
-- 2) then a running (window) SUM ordered by month.

WITH monthly_sales AS (
  SELECT
    p.product_id,
    p.product_name,
    DATE_FORMAT(o.order_date, 'yyyy-MM') AS sales_month,
    SUM(oi.quantity * oi.unit_price) AS monthly_sales
  FROM orders o
  JOIN order_items oi
    ON o.order_id = oi.order_id
  JOIN products p
    ON oi.product_id = p.product_id
  GROUP BY p.product_id, p.product_name, DATE_FORMAT(o.order_date, 'yyyy-MM')
)

SELECT
  product_id,
  product_name,
  sales_month,
  monthly_sales,
  SUM(monthly_sales) OVER (
    PARTITION BY product_id
    ORDER BY sales_month
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cumulative_sales
FROM monthly_sales
ORDER BY product_id, sales_month;

In [0]:
-- Second highest salary: rank distinct salaries with DENSE_RANK() and keep rank = 2.
-- (DENSE_RANK on salary treats ties as one rank, so 'second highest distinct salary' is rank 2.)

WITH ranked AS (
  SELECT
    salary,
    DENSE_RANK() OVER (ORDER BY salary DESC) AS salary_rank
  FROM employees
)

SELECT salary
FROM ranked
WHERE salary_rank = 2;

In [0]:
-- Customers who spent more in 2024 than in 2023:
-- 1) aggregate total spend per customer for each year (2023 and 2024 only),
-- 2) self-join the yearly aggregates on customer_id, one side per year,
-- 3) keep only customers whose 2024 spend exceeds 2023.

WITH yearly_spend AS (
  SELECT
    customer_id,
    YEAR(order_date) AS yr,
    SUM(amount) AS total_spent
  FROM orders
  WHERE YEAR(order_date) IN (2023, 2024)
  GROUP BY customer_id, YEAR(order_date)
)

SELECT
  y2023.customer_id,
  y2023.total_spent AS spend_2023,
  y2024.total_spent AS spend_2024,
  y2024.total_spent - y2023.total_spent AS spend_increase
FROM yearly_spend y2023
JOIN yearly_spend y2024
  ON y2023.customer_id = y2024.customer_id
 AND y2023.yr = 2023
 AND y2024.yr = 2024
WHERE y2024.total_spent > y2023.total_spent
ORDER BY spend_increase DESC;